# The Mathematical Foundations of XGBoost

This text reconstructs the comprehensive mathematical framework of Extreme Gradient Boosting (XGBoost). Our goal is to derive the complete mathematical formulation from first principles, utilizing a second-order Taylor expansion of the objective function. We will avoid hand-waving simplifications or intuitive leaps, explicitly defining every intermediate step, substitution, summation, and regularization boundary.

---

## 1. Core Architectural Differences and Motivation

XGBoost is an advanced variant of sequential ensemble modeling based on Gradient Boosting Machines (GBM). While traditional GBM algorithms rely on first-order gradients (the Jacobian vector) to optimize arbitrary differentiable loss functions, XGBoost employs a second-order framework that calculates the Hessian matrix to establish optimal splits and leaf weights.

### 1.1 Structural Divergence from Standard Decision Trees

A standard gradient-boosted decision tree relies on statistical metrics such as Gini Impurity, Shannon Entropy, or Mean Squared Error (MSE / Variance Reduction) to evaluate node splits. XGBoost, by contrast, introduces two core structural equations to guide tree construction and predict outputs:

1. **The Similarity Score Metric:** Used during node division to assess partition purity and structural quality.
2. **The Optimal Leaf Output Equation:** Used to compute numeric prediction values assigned to the terminal leaf nodes.

### 1.2 Mathematical Discrepancy by Task Type

Depending on whether the problem is a continuous regression or binary classification task, these metrics evaluate to distinct algebraic forms.

#### Case I: Continuous Regression Space

The similarity score for a generic node containing a set of instances $I$ is given by:

$$\text{Similarity Score}_{\text{Reg}} = \frac{\left( \sum_{i \in I} r_i \right)^2}{|I| + \lambda}$$

The output weight assigned to a terminal leaf containing data point indices $I$ is:

$$w_{\text{Reg}} = \frac{\sum_{i \in I} r_i}{|I| + \lambda}$$

Where:

* $r_i$ represents the continuous error residual associated with the $i$-th observation.
* $|I|$ denotes the total count of training instances within that specific node.
* $\lambda$ serves as the user-defined $L_2$ regularization hyperparameter.

#### Case II: Binary Classification Space

The similarity score transforms to incorporate underlying probability weights:

$$\text{Similarity Score}_{\text{Class}} = \frac{\left( \sum_{i \in I} r_i \right)^2}{\sum_{i \in I} \left[ p_i (1 - p_i) \right] + \lambda}$$

The output weight assigned to a classification terminal leaf node is:

$$w_{\text{Class}} = \frac{\sum_{i \in I} r_i}{\sum_{i \in I} \left[ p_i (1 - p_i) \right] + \lambda}$$

Where $p_i$ represents the probability prediction generated by the preceding ensemble stage for the $i$-th instance.

---

## 2. Setting Up the Global Objective Function

To understand where these four formulas originate, we must establish the formal multi-stage objective function.

### 2.1 The Additive Training Ensemble

Let $\hat{y}_i^{(t)}$ denote the cumulative prediction of the network for the $i$-th instance at iteration $t$. This prediction is constructed sequentially:

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)$$

Where:

* $\hat{y}_i^{(t-1)}$ is the static, frozen ensemble prediction accumulated from step $1$ to step $t-1$.
* $f_t(x_i)$ is the functional mapping of the newest independent tree learner being added at step $t$.

### 2.2 Constructing the Loss Function with Regularization

The global objective function $\mathcal{L}^{(t)}$ to be minimized at step $t$ combines an empirical loss metric $l$ across all $n$ data points with a structural regularization penalty $\Omega(f_t)$ across all $T$ leaves of the new tree:

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} l\left(y_i, \hat{y}_i^{(t)}\right) + \Omega(f_t)$$

Substituting our sequential additive prediction equation $\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)$ into the objective yields:

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} l\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)$$

### 2.3 Structural Complexity Penalty $\Omega(f)$

The regularization penalty explicitly constrains tree complexity to prevent overfitting. It penalizes both the total leaf count $T$ and the magnitude of the leaf weights $w$:

$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

Where:

* $\gamma$ acts as a penalty parameter for adding leaves, enforcing pruning.
* $\lambda$ enforces an $L_2$ shrinkage penalty on the terminal output weights vector $w$.

---

## 3. Second-Order Taylor Series Expansion

The objective function contains an arbitrary, potentially non-linear loss function $l$. To optimize it universally, we approximate it using a second-order Taylor series expansion.

### 3.1 The Mathematical Identity

Recall that the multi-variable Taylor expansion for a function $f(x + \Delta x)$ around a base point $x$ is given by:

$$f(x + \Delta x) \approx f(x) + f'(x)\Delta x + \frac{1}{2}f''(x)(\Delta x)^2$$

### 3.2 Mapping to the Loss Function

Let us define our variables to match this identity:

* The base point $x \implies \hat{y}_i^{(t-1)}$ (the fixed prediction from the previous step).
* The step change $\Delta x \implies f_t(x_i)$ (the new function optimization target).

Applying the expansion to our empirical loss function for an individual data point yields:

$$l\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) \approx l\left(y_i, \hat{y}_i^{(t-1)}\right) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)$$

Where:

* $g_i$ is the first-order derivative (gradient) of the loss function with respect to the prediction at step $t-1$:

$$g_i = \frac{\partial l(y_i, \hat{y}_i^{(t-1)})}{\partial \hat{y}_i^{(t-1)}}$$

* $h_i$ is the second-order derivative (Hessian) of the loss function with respect to the prediction at step $t-1$:

$$h_i = \frac{\partial^2 l(y_i, \hat{y}_i^{(t-1)})}{\partial \left(\hat{y}_i^{(t-1)}\right)^2}$$

### 3.3 Simplifying the Total Objective Function

Substituting this approximation back into our total objective function yields:

$$\mathcal{L}^{(t)} \approx \sum_{i=1}^{n} \left[ l\left(y_i, \hat{y}_i^{(t-1)}\right) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

Because the term $l\left(y_i, \hat{y}_i^{(t-1)}\right)$ depends solely on historical data and parameters from past iterations, it contains no active parameters from the current tree $f_t(x_i)$. It acts as a constant relative to $f_t$. Dropping this constant term simplifies our optimization objective to:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

---

## 4. Re-indexing via Leaf Memberships

A decision tree $f_t(x)$ operates by partitioning the feature space into disjoint regions and assigning a scalar weight $w_j$ to each region.

### 4.1 Transitioning from Instance Summation to Leaf Summation

Let $I_j = \{i \mid q(x_i) = j\}$ be the set of data point indices mapped to terminal leaf node $j$. Every instance $i$ falling into leaf $j$ receives the identical prediction weight:

$$f_t(x_i) = w_{q(x_i)} = w_j$$

We can rewrite the summation over individual training instances ($\sum_{i=1}^n$) as an outer summation over the tree's leaves ($\sum_{j=1}^T$) combined with an inner summation over the instances assigned to each leaf ($i \in I_j$):

$$\sum_{i=1}^{n} g_i f_t(x_i) = \sum_{j=1}^{T} \sum_{i \in I_j} g_i w_j = \sum_{j=1}^{T} w_j \left( \sum_{i \in I_j} g_i \right)$$

$$\sum_{i=1}^{n} \frac{1}{2} h_i f_t^2(x_i) = \sum_{j=1}^{T} \sum_{i \in I_j} \frac{1}{2} h_i w_j^2 = \sum_{j=1}^{T} \frac{1}{2} w_j^2 \left( \sum_{i \in I_j} h_i \right)$$

### 4.2 Grouping the Objective Function

Substituting these grouped expressions back into the simplified objective function $\tilde{\mathcal{L}}^{(t)}$ yields:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ w_j \left( \sum_{i \in I_j} g_i \right) + \frac{1}{2} w_j^2 \left( \sum_{i \in I_j} h_i \right) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

We can now factor out $w_j$ and $w_j^2$ inside a single unified summation across all $T$ leaves:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ \left(\sum_{i \in I_j} g_i\right) w_j + \frac{1}{2} \left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2 \right] + \gamma T$$

To clean up this notation, we define two aggregate statistics for each leaf $j$:

* $G_j = \sum_{i \in I_j} g_i$ (the sum of first-order gradients in leaf $j$)
* $H_j = \sum_{i \in I_j} h_i$ (the sum of second-order Hessians in leaf $j$)

This simplifies the core objective function to:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ G_j w_j + \frac{1}{2}(H_j + \lambda)w_j^2 \right] + \gamma T$$

---

## 5. Deriving the Optimal Leaf Weights and Optimal Loss

This structural equation allows us to find the optimal prediction weight for each leaf independently, as the choices for $w_j$ are decoupled across the disjoint leaf sets.

### 5.1 Optimization via Differentiation

For a given leaf node $j$, the local cost function is a simple quadratic equation with respect to $w_j$:

$$\phi(w_j) = G_j w_j + \frac{1}{2}(H_j + \lambda)w_j^2$$

To find the value of $w_j$ that minimizes this cost, we take the partial derivative with respect to $w_j$ and set it to zero:

$$\frac{\partial \phi(w_j)}{\partial w_j} = \frac{\partial}{\partial w_j} \left[ G_j w_j + \frac{1}{2}(H_j + \lambda)w_j^2 \right] = 0$$

$$G_j + \frac{1}{2}(H_j + \lambda) \cdot 2w_j = 0$$

$$G_j + (H_j + \lambda)w_j = 0$$

$$(H_j + \lambda)w_j = -G_j$$

Solving for $w_j$ yields the equation for the **Optimal Leaf Weight**:

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

### 5.2 Deriving the Minimal Evaluated Objective Value

To find the minimum possible loss for a given tree structure, we substitute this optimal weight $w_j^*$ back into our objective function $\tilde{\mathcal{L}}^{(t)}$:

$$\tilde{\mathcal{L}}^{(t)*} = \sum_{j=1}^{T} \left[ G_j \left( -\frac{G_j}{H_j + \lambda} \right) + \frac{1}{2}(H_j + \lambda)\left( -\frac{G_j}{H_j + \lambda} \right)^2 \right] + \gamma T$$

$$\tilde{\mathcal{L}}^{(t)*} = \sum_{j=1}^{T} \left[ -\frac{G_j^2}{H_j + \lambda} + \frac{1}{2}(H_j + \lambda) \frac{G_j^2}{(H_j + \lambda)^2} \right] + \gamma T$$

Canceling out the linear $(H_j + \lambda)$ term in the second component yields:

$$\tilde{\mathcal{L}}^{(t)*} = \sum_{j=1}^{T} \left[ -\frac{G_j^2}{H_j + \lambda} + \frac{1}{2} \frac{G_j^2}{H_j + \lambda} \right] + \gamma T$$

Combining these two fractions ($1 - \frac{1}{2} = \frac{1}{2}$) gives us the standard **Optimal Objective Loss** equation:

$$\tilde{\mathcal{L}}^{(t)*} = -\frac{1}{2} \sum_{j=1}^{T} \left( \frac{G_j^2}{H_j + \lambda} \right) + \gamma T$$

---

## 6. Concrete Specializations for Core ML Tasks

The general equations for $w_j^*$ and $\tilde{\mathcal{L}}^{(t)*}$ apply to any twice-differentiable loss function. Let us now derive the specific metrics used for standard regression and classification tasks.

### 6.1 Task Domain I: Continuous Regression with MSE

For standard continuous regression, we use the Mean Squared Error (MSE) loss function:

$$l(y_i, \hat{y}_i) = \frac{1}{2}(y_i - \hat{y}_i)^2$$

#### Step 1: Compute the Gradient ($g_i$)

We apply the chain rule to take the derivative with respect to the prediction value $\hat{y}_i$:

$$g_i = \frac{\partial}{\partial \hat{y}_i} \left[ \frac{1}{2}(y_i - \hat{y}_i)^2 \right] = \frac{1}{2} \cdot 2(y_i - \hat{y}_i) \cdot (-1) = -(y_i - \hat{y}_i)$$

This confirms that the gradient is simply the negative error residual:

$$g_i = \hat{y}_i - y_i = -r_i$$

#### Step 2: Compute the Hessian ($h_i$)

We differentiate the gradient $g_i$ a second time with respect to $\hat{y}_i$:

$$h_i = \frac{\partial g_i}{\partial \hat{y}_i} = \frac{\partial}{\partial \hat{y}_i} (\hat{y}_i - y_i) = 1$$

#### Step 3: Substitute into the Core Framework Equations

Now we compute our aggregate leaf statistics by summing over the instances in leaf $j$:

$$G_j = \sum_{i \in I_j} g_i = \sum_{i \in I_j} (-r_i) = -\sum_{i \in I_j} r_i$$

$$H_j = \sum_{i \in I_j} h_i = \sum_{i \in I_j} 1 = |I_j|$$

Substituting $G_j$ and $H_j$ into our optimal leaf weight formula $w_j^* = -\frac{G_j}{H_j + \lambda}$ yields:

$$w_j^* = -\frac{-\sum_{i \in I_j} r_i}{|I_j| + \lambda} = \frac{\sum_{i \in I_j} r_i}{|I_j| + \lambda}$$

This matches our original regression weight formula. The corresponding similarity score represents the reduction in loss contributed by leaf $j$ (ignoring the negative multiplier and the scalar fraction $\frac{1}{2}$):

$$\text{Similarity Score}_{\text{Reg}} = \frac{G_j^2}{H_j + \lambda} = \frac{\left(-\sum_{i \in I_j} r_i\right)^2}{|I_j| + \lambda} = \frac{\left(\sum_{i \in I_j} r_i\right)^2}{|I_j| + \lambda}$$

---

### 6.2 Task Domain II: Binary Classification with Log-Loss

For binary classification tasks where $y_i \in \{0, 1\}$, we use the binary cross-entropy (Log-Loss) function. To map real-valued tree predictions back to probabilities, we apply the logistic sigmoid function to our ensemble prediction: $p_i = \sigma(\hat{y}_i) = \frac{1}{1 + e^{-\hat{y}_i}}$.

The log-loss cost function is defined as:

$$l(y_i, \hat{y}_i) = - \left[ y_i \ln(p_i) + (1 - y_i) \ln(1 - p_i) \right]$$

#### Step 1: Compute the Gradient ($g_i$)

First, we note the derivative identity for the logistic function:

$$\frac{\partial p_i}{\partial \hat{y}_i} = p_i(1 - p_i)$$

Using this identity and the chain rule, we differentiate the loss function with respect to $\hat{y}_i$:

$$g_i = -\left[ y_i \frac{1}{p_i} \cdot \frac{\partial p_i}{\partial \hat{y}_i} + (1 - y_i)\frac{1}{1 - p_i} \cdot \left(-\frac{\partial p_i}{\partial \hat{y}_i}\right) \right]$$

$$g_i = -\left[ y_i \frac{1}{p_i} p_i(1 - p_i) - (1 - y_i)\frac{1}{1 - p_i} p_i(1 - p_i) \right]$$

Canceling out the terms in the fractions simplifies the equation to:

$$g_i = -\left[ y_i(1 - p_i) - (1 - y_i)p_i \right] = -\left[ y_i - y_i p_i - p_i + y_i p_i \right]$$

$$g_i = -\left[ y_i - p_i \right] = p_i - y_i$$

This confirms that the gradient is the difference between the predicted probability and the true binary target. This can also be expressed as the negative classification residual: $g_i = -r_i$.

#### Step 2: Compute the Hessian ($h_i$)

Next, we differentiate the gradient $g_i = p_i - y_i$ a second time with respect to $\hat{y}_i$:

$$h_i = \frac{\partial g_i}{\partial \hat{y}_i} = \frac{\partial}{\partial \hat{y}_i}(p_i - y_i) = \frac{\partial p_i}{\partial \hat{y}_i} = p_i(1 - p_i)$$

#### Step 3: Substitute into the Core Framework Equations

We sum these individual instance derivatives to compute our aggregate leaf statistics for leaf $j$:

$$G_j = \sum_{i \in I_j} g_i = \sum_{i \in I_j} (p_i - y_i) = -\sum_{i \in I_j} r_i$$

$$H_j = \sum_{i \in I_j} h_i = \sum_{i \in I_j} p_i(1 - p_i)$$

Substituting $G_j$ and $H_j$ into our optimal weight equation $w_j^* = -\frac{G_j}{H_j + \lambda}$ yields:

$$w_j^* = -\frac{-\sum_{i \in I_j} r_i}{\sum_{i \in I_j} p_i(1 - p_i) + \lambda} = \frac{\sum_{i \in I_j} r_i}{\sum_{i \in I_j} p_i(1 - p_i) + \lambda}$$

This matches our classification leaf weight formula. The corresponding classification similarity score is:

$$\text{Similarity Score}_{\text{Class}} = \frac{G_j^2}{H_j + \lambda} = \frac{\left(\sum_{i \in I_j} r_i\right)^2}{\sum_{i \in I_j} p_i(1 - p_i) + \lambda}$$

---

## 7. Node Splitting and Loss Reduction

To grow a tree, we must choose how to split a node into left ($L$) and right ($R$) children. We evaluate potential split points by measuring how much they reduce the overall objective loss.

### 7.1 Deriving the Gain Formula

When a parent node is split, its instances are partitioned into left and right child nodes. The reduction in loss (or Gain) from this split is defined as the total optimal loss of the child nodes subtracted from the optimal loss of the unpartitioned parent node:

$$\text{Gain} = \text{Loss}_{\text{split}} - \text{Loss}_{\text{parent}}$$

$$\text{Gain} = \left[ \tilde{\mathcal{L}}^*_{\text{Left}} + \tilde{\mathcal{L}}^*_{\text{Right}} \right] - \tilde{\mathcal{L}}^*_{\text{Parent}}$$

Substituting our optimal loss equation $\tilde{\mathcal{L}}^{(t)*} = -\frac{1}{2} \sum \left( \frac{G_j^2}{H_j + \lambda} \right) + \gamma T$ into this definition yields:

$$\text{Gain} = \left[ -\frac{1}{2}\frac{G_L^2}{H_L + \lambda} + \gamma + \left(-\frac{1}{2}\frac{G_R^2}{H_R + \lambda}\right) + \gamma \right] - \left[ -\frac{1}{2}\frac{G_P^2}{H_P + \lambda} + \gamma \right]$$

Factoring out the constant multiplier $\frac{1}{2}$ and simplifying the complexity penalties ($1\gamma + 1\gamma - 1\gamma = \gamma$) gives us the standard **XGBoost Gain Split Equation**:

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{G_P^2}{H_P + \lambda} \right] - \gamma$$

Where:

* $G_L, H_L$ are the aggregate gradients and Hessians of the proposed left child node.
* $G_R, H_R$ are the aggregate gradients and Hessians of the proposed right child node.
* $G_P, H_P$ are the aggregate gradients and Hessians of the original parent node ($G_P = G_L + G_R$ and $H_P = H_L + H_R$).
* $\gamma$ serves as a baseline threshold: a split is only accepted if the calculated gain is positive, meaning the raw reduction in loss exceeds the cost of adding a new leaf node.

---

## 8. Summary Blueprint

### Master Formula Reference Table

| Formula Component | General Form | Continuous Regression Space | Binary Classification Space |
| --- | --- | --- | --- |
| **Instance Gradient ($g_i$)** | $\frac{\partial l(y_i, \hat{y}_i^{(t-1)})}{\partial \hat{y}_i^{(t-1)}}$ | $\hat{y}_i - y_i = -r_i$ | $p_i - y_i = -r_i$ |
| **Instance Hessian ($h_i$)** | $\frac{\partial^2 l(y_i, \hat{y}_i^{(t-1)})}{\partial (\hat{y}_i^{(t-1)})^2}$ | $1$ | $p_i(1 - p_i)$ |
| **Similarity Score** | $\frac{G_j^2}{H_j + \lambda}$ | $\frac{\left(\sum r_i\right)^2}{\vert I_j \vert + \lambda}$ | $\frac{\left(\sum r_i\right)^2}{\sum p_i(1 - p_i) + \lambda}$ |
| **Optimal Leaf Output ($w_j^*$)** | $-\frac{G_j}{H_j + \lambda}$ | $\frac{\sum r_i}{\vert I_j \vert + \lambda}$ | $\frac{\sum r_i}{\sum p_i(1 - p_i) + \lambda}$ |

### Complete Step-by-Step Derivation Logic

```
[Global Loss Objective Function]
               │
               ▼  (Apply 2nd-Order Taylor Series Expansion)
[ Taylor Multi-Variable Approximation ]
               │
               ▼  (Drop historical constant terms)
[ Streamlined Loss Objective Function ]
               │
               ▼  (Re-index from instances to leaf groupings)
[ Decoupled Structural Leaf Form ]
               │
               ▼  (Differentiate with respect to weight vector w)
[ Optimal Weight & Minimal Loss Equations ]
               │
               ▼  (Compute child vs. parent loss differences)
[ Final Master Gain Split Criterion ]
```